# AI City 2026 Track 5 — SEINE inference (Colab A100)

End-to-end: download data layout check → build index → SEINE inference → validate → zip for submission.

Runtime: **GPU (A100)**. Run cells top to bottom; edit the **CONFIG** cell paths to your Drive.


## 1. Mount Drive & GPU check


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code (SEINE + this repo)

Set `AICITY_GIT` to your repo URL, **or** set `AICITY_SRC` to a Drive copy of `aicity_track5/`.


In [ ]:
import os
%cd /content
# --- option A: clone your aicity_track5 repo ---
AICITY_GIT = ''   # e.g. 'https://github.com/<you>/aicity_track5.git'
# --- option B: copy from Drive (used if AICITY_GIT is empty) ---
AICITY_SRC = '/content/drive/MyDrive/aicity_track5'

if AICITY_GIT:
    !git clone $AICITY_GIT /content/aicity_track5
else:
    !cp -r "$AICITY_SRC" /content/aicity_track5

!git clone https://github.com/Vchitect/SEINE /content/third_party/SEINE
%cd /content/aicity_track5
print('repo ready')

## 3. Install dependencies

Colab ships a recent torch; we install SEINE's lighter deps + the eval stack. If the xformers wheel does not match Colab's torch, set `enable_xformers_memory_efficient_attention: False` in the config (cell 6).


In [ ]:
!pip install -q omegaconf einops natsort decord imageio timm \
    rotary-embedding-torch==0.3.5 'diffusers==0.15.0' 'transformers==4.29.2' Pillow
# eval metrics
!pip install -q -r requirements-eval.txt
# (optional) xformers — comment out if it forces a torch downgrade
# !pip install -q xformers

## 4. Download model weights → pretrained/

`seine.pt` (SEINE checkpoint) + `stable-diffusion-v1-4` (VAE/UNet config/text encoder).


In [ ]:
!mkdir -p pretrained
!pip install -q gdown huggingface_hub
# SEINE checkpoint (google drive folder)
!gdown --fuzzy 'https://drive.google.com/drive/folders/1cWfeDzKJhpb0m6HA5DoMOH0_ItuUY95b' -O pretrained --folder
# Stable Diffusion v1-4 base
!huggingface-cli download CompVis/stable-diffusion-v1-4 --local-dir pretrained/stable-diffusion-v1-4
!ls -lh pretrained pretrained/stable-diffusion-v1-4

## 5. CONFIG — set your Drive data paths

Point these at the **mounted** WTS test (and val) folders you downloaded.


In [ ]:
# EDIT THESE:
TEST_ROOT = '/content/drive/MyDrive/WTS_TV2V/test'   # contains <sample>/input/*.png (+ caption.json)
VAL_ROOT  = '/content/drive/MyDrive/WTS_TV2V/val'    # set to '' if you only run test
SEINE_ROOT = '/content/third_party/SEINE'
CONFIG = 'configs/seine_track5.yaml'
import os; print('test exists:', os.path.isdir(TEST_ROOT))

## 6. Inspect the real test layout (IMPORTANT)

Confirm where `caption.json` lives, history-frame count (→ `cond_frames`), resolution, and the `frame length` (N) distribution. `build_index.py` already handles caption inside OR beside `input/`.


In [ ]:
!python data/inspect_dataset.py --data_root "$TEST_ROOT" --max_items 8

## 7. Build the index (test + val)


In [ ]:
!python data/build_index.py --data_root "$TEST_ROOT" --split test --output data/index_test.jsonl
if VAL_ROOT:
    !python data/build_index.py --data_root "$VAL_ROOT" --split val --output data/index_val.jsonl
!head -c 800 data/index_test.jsonl; print()

## 8. Smoke test — 2 samples, then format check

Verifies SEINE loads, generates, resizes, and writes the right PNG layout.


In [ ]:
!python baselines/seine_infer.py --config $CONFIG --index data/index_test.jsonl \
    --output outputs/seine --seine_root $SEINE_ROOT --limit 2
!python submission/validate_submission.py --index data/index_test.jsonl --pred_dir outputs/seine || true
# (validate flags the not-yet-generated samples — expected with --limit)

## 9. (Optional) Evaluate on val to tune the config

Needs `VAL_ROOT`. Sweep `image_size`, `cond_frames`, `cfg_scale`, `num_sampling_steps` in the config and re-run to compare metrics.


In [ ]:
if VAL_ROOT:
    !python baselines/seine_infer.py --config $CONFIG --index data/index_val.jsonl \
        --output outputs/seine_val --seine_root $SEINE_ROOT --limit 10
    !python eval/eval_metrics.py --index data/index_val.jsonl --pred_dir outputs/seine_val \
        --metrics psnr,ssim,lpips,clip,fvd --limit 10 --output outputs/seine_val/metrics.json

## 10. Full test inference

Resumable — re-running skips samples that already have all N frames (omit `--overwrite`). Long run; keep the tab alive.


In [ ]:
!python baselines/seine_infer.py --config $CONFIG --index data/index_test.jsonl \
    --output outputs/seine --seine_root $SEINE_ROOT

## 11. Validate + zip for submission


In [ ]:
!python submission/validate_submission.py --index data/index_test.jsonl --pred_dir outputs/seine
!python submission/make_zip.py --pred_dir outputs/seine --zip_path outputs/submission_seine.zip
!ls -lh outputs/submission_seine.zip

## 12. Download / save submission

The zip root contains `<sample>/0.png .. (N-1).png` (no extra wrapper) — the Track 5 format.


In [ ]:
# save to Drive
!cp outputs/submission_seine.zip /content/drive/MyDrive/
# or download directly
from google.colab import files; files.download('outputs/submission_seine.zip')